# VC Deal Flow Signal — Startup Engineering Velocity Analysis

**Dataset:** [VC Deal Flow Signal](https://www.kaggle.com/datasets/thedatanerd2026/vc-deal-flow-signal)

**License:** CC BY 4.0

**Author:** The Data Nerd

This notebook analyzes engineering velocity across 324+ venture-backed startups in 15 sectors. The core insight: when a startup's commit velocity suddenly 3x's and contributor count doubles, that startup is likely raising within 3-6 weeks.

**Live dashboard:** [signals.gitdealflow.com](https://signals.gitdealflow.com)

**Paper:** [SSRN 6606558](https://ssrn.com/abstract=6606558)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from datetime import datetime

plt.style.use('ggplot')
sns.set_palette('viridis')

print('Setup complete!')

## 1. Load the Dataset

We fetch the latest data directly from the live API at `signals.gitdealflow.com`.

In [ ]:
# Fetch live data from the API
resp = requests.get('https://signals.gitdealflow.com/api/signals.json', timeout=10)
data = resp.json()

# Flatten into a dataframe
records = []
for sector in data['sectors']:
    for startup in sector['startups']:
        startup['sector'] = sector['name']
        startup['sector_slug'] = sector['slug']
        startup['total_sector_startups'] = sector['startupCount']
        records.append(startup)

df = pd.DataFrame(records)
print(f'Loaded {len(df)} startups from {len(data["sectors"])} sectors')
print(f'Period: {data["meta"]["period"]["name"]}')
df.head()

In [ ]:
# Parse numeric columns
df['commitVelocity14d'] = pd.to_numeric(df['commitVelocity14d'], errors='coerce')
df['contributors'] = pd.to_numeric(df['contributors'], errors='coerce')
df['newRepos'] = pd.to_numeric(df['newRepos'], errors='coerce')
df['contributorGrowth'] = pd.to_numeric(df['contributorGrowth'].str.replace('%','').str.replace('+',''), errors='coerce')
df['commitVelocityChange'] = pd.to_numeric(df['commitVelocityChange'].str.replace('%','').str.replace('+',''), errors='coerce')

print('Data types after cleaning:')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')

## 2. Sector Analysis

Which sectors have the most engineering activity? Let's look at total startups per sector and average velocity.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sector counts
sector_counts = df['sector'].value_counts()
axes[0].barh(range(len(sector_counts)), sector_counts.values, color=sns.color_palette('viridis', len(sector_counts)))
axes[0].set_yticks(range(len(sector_counts)))
axes[0].set_yticklabels(sector_counts.index)
axes[0].set_xlabel('Number of Startups')
axes[0].set_title('Startups by Sector')

# Average velocity by sector
avg_vel = df.groupby('sector')['commitVelocity14d'].mean().sort_values(ascending=True)
axes[1].barh(range(len(avg_vel)), avg_vel.values, color=sns.color_palette('magma', len(avg_vel)))
axes[1].set_yticks(range(len(avg_vel)))
axes[1].set_yticklabels(avg_vel.index)
axes[1].set_xlabel('Avg Commits / 14 Days')
axes[1].set_title('Average Engineering Velocity by Sector')

plt.tight_layout()
plt.show()

## 3. Signal Type Distribution

The dataset captures different acceleration signals. Which ones dominate?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

signal_counts = df['signalType'].value_counts()
colors = plt.cm.Set2(range(len(signal_counts)))

wedges, texts, autotexts = ax.pie(
    signal_counts.values,
    labels=signal_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90
)
ax.set_title('Signal Type Distribution Across 324+ Startups')

plt.show()

print('\nSignal breakdown:')
for sig, count in signal_counts.items():
    print(f'  {sig}: {count} startups ({count/len(df)*100:.1f}%)')

## 4. Geographic Distribution

Where are the most active startups located?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

geo_counts = df['geography'].value_counts().head(10)
ax.bar(range(len(geo_counts)), geo_counts.values, color=sns.color_palette('cubehelix', len(geo_counts)))
ax.set_xticks(range(len(geo_counts)))
ax.set_xticklabels(geo_counts.index, rotation=45, ha='right')
ax.set_ylabel('Startup Count')
ax.set_title('Top 10 Startup Locations')

plt.tight_layout()
plt.show()

## 5. Top Accelerating Startups

Which startups show the highest engineering velocity changes?

In [ ]:
top_by_change = df.nlargest(10, 'commitVelocityChange')[['name', 'sector', 'commitVelocityChange', 'commitVelocity14d', 'signalType', 'geography']]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(range(len(top_by_change)), top_by_change['commitVelocityChange'].values)
ax.set_yticks(range(len(top_by_change)))
ax.set_yticklabels(top_by_change['name'].values)
ax.set_xlabel('Velocity Change (%)')
ax.set_title('Top 10 Startups by Engineering Velocity Acceleration')

for i, (_, row) in enumerate(top_by_change.iterrows()):
    ax.text(row['commitVelocityChange'] + 20, i, f'{row["sector"]} · {row["signalType"]}',
            va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\nTop 10 accelerating startups:')
display(top_by_change)

## 6. Correlation Analysis

Do startups with more contributors also have more commits? Let's check.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

corr_cols = ['commitVelocity14d', 'contributors', 'newRepos', 'commitVelocityChange', 'contributorGrowth']
corr_matrix = df[corr_cols].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Engineering Metrics Correlation Matrix')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

scatter = ax.scatter(df['commitVelocity14d'], df['contributors'],
                     c=df['commitVelocityChange'].fillna(0), cmap='viridis',
                     alpha=0.6, s=80)
plt.colorbar(scatter, ax=ax, label='Velocity Change (%)')
ax.set_xlabel('Commits / 14 Days')
ax.set_ylabel('Contributors')
ax.set_title('Engineering Activity Landscape')

# Annotate a few standouts
standouts = df.nlargest(5, 'commitVelocity14d')[['name', 'commitVelocity14d', 'contributors']]
for _, row in standouts.iterrows():
    ax.annotate(row['name'], (row['commitVelocity14d'], row['contributors']),
                xytext=(5, 5), textcoords='offset points', fontsize=8)

plt.tight_layout()
plt.show()

## 7. Stage Distribution

At what stage do startups show the most engineering acceleration?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stage_counts = df['stage'].value_counts()
axes[0].pie(stage_counts.values, labels=stage_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set3', len(stage_counts)), startangle=90)
axes[0].set_title('Stage Distribution')

stage_vel = df.groupby('stage')['commitVelocityChange'].mean().sort_values()
axes[1].barh(range(len(stage_vel)), stage_vel.values, color=sns.color_palette('viridis', len(stage_vel)))
axes[1].set_yticks(range(len(stage_vel)))
axes[1].set_yticklabels(stage_vel.index)
axes[1].set_xlabel('Avg Velocity Change (%)')
axes[1].set_title('Average Acceleration by Stage')

plt.tight_layout()
plt.show()

## 8. Key Takeaways

1. **Average engineering velocity varies dramatically by sector** — some sectors show 5x the commit activity of others
2. **Deploy frequency spikes dominate** — most accelerated startups are shipping faster, not necessarily hiring more
3. **Pre-seed and seed-stage startups show the highest volatility** — early-stage engineering teams swing from 0 to rapid acceleration
4. **Geographic concentration** — activity clusters in traditional tech hubs (US, particularly SF Bay Area)
5. **Velocity change and contributor count show moderate correlation** — growing teams ship more, but the relationship isn't linear

### How to use this data
- **VCs & angels:** Filter by sector for deal sourcing signals
- **Founders:** Benchmark your startup's velocity against sector averages
- **Researchers:** Download the full dataset for academic analysis
- **AI agents:** Use the MCP server at `npx -y @gitdealflow/mcp-signal`

### Links
- [Live Dashboard](https://signals.gitdealflow.com)
- [StartUp Grader — grade any GitHub org](https://gitdealflow.com/grader)
- [Scout Score — check your GitHub taste](https://gitdealflow.com/scout)
- [State of Startup Engineering Report](https://gitdealflow.com/state-of-startup-engineering-2026-q3)
- [SSRN Methodology Paper](https://ssrn.com/abstract=6606558)
- [MCP Server on npm](https://www.npmjs.com/package/@gitdealflow/mcp-signal)

---
**License:** CC BY 4.0 — Free to use, share, and adapt with attribution.